In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from rl4co.utils.trainer import RL4COTrainer

sys.path.append(str(Path('..').resolve() / 'src'))

from dvrptw_bench.data.instance_filters import find_rc_instances
from dvrptw_bench.data.solomon_parser import parse_solomon
from dvrptw_bench.metrics.objective import total_distance
from dvrptw_bench.rl.env_dynamic import MTVRPDynamicEnv
from dvrptw_bench.rl.routefinder_adapter import instance_to_routefinder_td, routefinder_actions_to_solution
from dvrptw_bench.viz.route_plot import plot_routes
from routefinder.envs.mtvrp import MTVRPGenerator
from routefinder.models import RouteFinderBase, RouteFinderPolicy
from routefinder.utils import evaluate as evaluate_routefinder


In [2]:
DATASET_ROOT = Path('../dataset/solomon_rc100')
OUTPUT_ROOT = Path('../outputs/notebook_routefinder_dynamic_10')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NUM_CUSTOMERS = 10
NUM_EPOCHS = 3
BATCH_SIZE = 256
TRAIN_DATA_SIZE = 2_000
VAL_DATA_SIZE = 256
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-6
NUM_AUGMENT = 8
MAX_EVAL_INSTANCES = 5
NORMALIZE_COORDS = True
DOD = 0.3
CUTOFF_RATIO = 0.8
DYNAMIC_SEED = 1234
LATENESS_PENALTY = 1.0

if torch.cuda.is_available():
    device = torch.device('cuda')
    accelerator = 'gpu'
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
    accelerator = 'mps'
else:
    device = torch.device('cpu')
    accelerator = 'cpu'

print('Device:', device)
print('Dataset root:', DATASET_ROOT.resolve())
print('Output root:', OUTPUT_ROOT.resolve())


Device: mps
Dataset root: /Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/dataset/solomon_rc100
Output root: /Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/outputs/notebook_routefinder_dynamic_10


In [ ]:
generator = MTVRPGenerator(num_loc=NUM_CUSTOMERS, max_time=10.11, variant_preset='vrptw')
env = MTVRPDynamicEnv(
    generator=generator,
    check_solution=False,
    dod=DOD,
    cutoff_time=CUTOFF_RATIO,
    allow_late_customers=True,
    lateness_penalty=LATENESS_PENALTY,
    dynamic_seed=DYNAMIC_SEED,
)
policy = RouteFinderPolicy(env_name=env.name).to(device)
model = RouteFinderBase(
    env,
    policy,
    batch_size=BATCH_SIZE,
    train_data_size=TRAIN_DATA_SIZE,
    val_data_size=VAL_DATA_SIZE,
    optimizer_kwargs={'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY},
)

trainer = RL4COTrainer(
    max_epochs=NUM_EPOCHS,
    accelerator=accelerator,
    devices=1,
    logger=None,
    num_sanity_val_steps=0,
    precision='32-true',
)
trainer.fit(model)
trainer.save_checkpoint(str(OUTPUT_ROOT / 'routefinder_dynamic_10.ckpt'))


/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to poten

┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ env      │ MTVRPDynamicEnv   │      0 │ train │     0 │
│ 1 │ policy   │ RouteFinderPolicy │  1.3 M │ train │     0 │
│ 2 │ baseline │ SharedBaseline    │      0 │ train │     0 │
└───┴──────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 112                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.v

SystemExit: 1

/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
instance_paths = find_rc_instances(DATASET_ROOT)
assert instance_paths, f'No RC instances found under {DATASET_ROOT.resolve()}'
if MAX_EVAL_INSTANCES is not None:
    instance_paths = instance_paths[:MAX_EVAL_INSTANCES]

instances = [parse_solomon(path, max_customers=NUM_CUSTOMERS) for path in instance_paths]
print('Loaded instances:', [instance.instance_id for instance in instances])


def rollout_actions(env, td_reset, actions):
    state = td_reset.clone()
    action_seq = actions[0].detach().cpu().tolist()
    for token in action_seq:
        state.update({'action': torch.tensor([int(token)], device=state.device)})
        state = env._step(state)
        if bool(state['done'].item()):
            break
    return state


def solve_with_dynamic_routefinder(instance, num_augment=NUM_AUGMENT):
    t0 = time.perf_counter()
    td = instance_to_routefinder_td(instance, normalize_coords=NORMALIZE_COORDS).to(device)
    td_reset = env.reset(td)
    model.to(device).eval()

    with torch.inference_mode():
        if num_augment > 1:
            out = evaluate_routefinder(model, td_reset.clone(), num_augment=num_augment)
            actions = out.get('best_aug_actions', out.get('best_multistart_actions', out.get('actions')))
        else:
            out = model.policy(
                td_reset.clone(),
                env,
                phase='test',
                decode_type='greedy',
                return_actions=True,
            )
            actions = out['actions']

    td_final = rollout_actions(env, td_reset, actions)
    solution = routefinder_actions_to_solution(actions, instance, strategy='routefinder-dynamic')
    solution.total_distance = total_distance(instance, solution)
    solution.solve_time_s = time.perf_counter() - t0
    solution.details.update(
        {
            'num_augment': num_augment,
            'dod': DOD,
            'cutoff_ratio': CUTOFF_RATIO,
            'hidden_initial': int((~td_reset['revealed'][0, 1:]).sum().item()),
            'tardiness': float(td_final['total_tardiness'][0, 0].item()),
        }
    )
    return solution, td_reset, td_final


rows = []
routefinder_solutions = {}
for instance in instances:
    solution, td_reset, td_final = solve_with_dynamic_routefinder(instance)
    objective = solution.total_distance + LATENESS_PENALTY * float(td_final['total_tardiness'][0, 0].item())
    routefinder_solutions[instance.instance_id] = solution
    rows.append(
        {
            'instance_id': instance.instance_id,
            'n_customers': instance.n_customers,
            'dynamic_customers': int(td_reset['is_dynamic'][0, 1:].sum().item()),
            'hidden_initial': int((~td_reset['revealed'][0, 1:]).sum().item()),
            'distance': solution.total_distance,
            'tardiness': float(td_final['total_tardiness'][0, 0].item()),
            'objective': objective,
            'routes': len([r for r in solution.routes if r.node_ids]),
            'solve_time_s': solution.solve_time_s,
        }
    )

results_df = pd.DataFrame(rows).sort_values('objective').reset_index(drop=True)
results_df[['distance', 'tardiness', 'objective', 'solve_time_s']] = results_df[
    ['distance', 'tardiness', 'objective', 'solve_time_s']
].round(3)

display(results_df)
display(results_df[['distance', 'tardiness', 'objective', 'solve_time_s']].mean().to_frame('mean').T.round(3))
# results_df.to_csv(OUTPUT_ROOT / 'routefinder_dynamic_eval_10.csv', index=False)
print('Saved table to', OUTPUT_ROOT / 'routefinder_dynamic_eval_10.csv')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_df = results_df.set_index('instance_id')
plot_df['objective'].plot(kind='bar', ax=axes[0], color='tab:blue')
axes[0].set_title('Dynamic Objective by Instance')
axes[0].set_ylabel('Distance + lateness penalty')
axes[0].grid(axis='y', alpha=0.3)

plot_df[['distance', 'tardiness']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Distance and Tardiness')
axes[1].set_ylabel('Value')
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
# fig.savefig(OUTPUT_ROOT / 'routefinder_dynamic_eval_10.png', dpi=180)
plt.show()
print('Saved summary plot to', OUTPUT_ROOT / 'routefinder_dynamic_eval_10.png')

for instance in instances[: min(3, len(instances))]:
    print(f'\n=== {instance.instance_id} ===')
    solution = routefinder_solutions[instance.instance_id]
    print('Distance:', solution.total_distance)
    print('Details:', solution.details)
    plot_routes(instance, solution)
